In [534]:
import json

import numpy as np
import pandas as pd
import talib
import plotly.graph_objects as go
import plotly.subplots as sub
import yfinance
from pandas import DataFrame
from termcolor import cprint

# md = yfinance.download(
#     "EURCHF=X",
#     start="2020-01-02",
#     end="2020-01-03",
#     interval="1h",
#     progress=False,
# )

md = yfinance.download(
    "GC=F",
    start="2020-01-12",
    end="2020-04-20",
    interval="1h",
    progress=False,
)
# print(md)

##########################################

# base_path = "~/projects/life/exante_history/data"
# path = f"{base_path}/forex/live-EUR:CHF.E.FX-quotes-60.jsonl"
# path = f"{base_path}/forex/small.jsonl"
#
# # print(md.head())
#
# md = pd.read_json(path, orient='records', lines=True)
# md.columns = ["timestamp", "Open", "High", "Low", "Close"]
# md.set_index('timestamp', inplace=True)
# md.sort_index(inplace=True)
#
# ohlc_dict = {'Open': 'first', 'High': 'max', 'Low': 'min', 'Close': 'last'}
# md = md.resample('1h', closed='left', label='left').apply(ohlc_dict)
# md.dropna(inplace=True)


##########################################


# md = yfinance.download(
#     "COPX",
#     start="2021-02-01",
#     end="2021-04-01",
#     interval="1h",
#     progress=False,
# )

price = go.Candlestick(
    x=md.index,
    open=md['Open'],
    high=md['High'],
    low=md['Low'],
    close=md['Close'],
    hoverinfo='skip',
    increasing=go.candlestick.Increasing(
        line=dict(width=1, color="#0a9"),
        fillcolor='#0a9',
    ),
    decreasing=go.candlestick.Decreasing(
        line=dict(width=1, color="#f44"),
        fillcolor='#f44',
    )
)

layout = go.Layout(
    # template='plotly_white',
    width=800,
    height=500,
    autosize=False,
    margin=dict(t=30, b=30, l=60, r=40),
    showlegend=False,
    paper_bgcolor='#f5f5f5',
    plot_bgcolor='#fff',
    xaxis=dict(
        linecolor="#ddd",
        # showspikes=True,
        # # Format spike
        # spikethickness=1,
        # spikedash="dot",
        # spikecolor="#999999",
        # spikemode="across",
        # grid
        showgrid=True,
        gridcolor="#ddd",
        # fixedrange=True,
        showticklabels=False,
        tickformat="%H~%M~%S.%2f",
        ticklabelposition="inside left",
        ticks="inside",
        tickwidth=10,
    ),
    yaxis=dict(
        linecolor="#ddd",
        # grid
        showgrid=True,
        gridcolor="#ddd",
        fixedrange=False,
        tickformat=".4f"
    ),
    font=dict(
        family="Menlo, monospace",
        size=11,
        color="#444"
    )
)


In [535]:
fig = sub.make_subplots(
    figure=go.Figure(layout=layout),
    rows=2,
    cols=1,
    shared_xaxes=True,
    shared_yaxes=False,
    vertical_spacing=0.03,
    row_heights=[0.75, 0.25],
    # subplot_titles=["Price", "Indicators"],
)

fig.add_trace(price, 1, 1)

# fig.layout.update(yaxis3=go.YAxis())
# fig.layout.update(yaxis3=go.YAxis(overlaying='y', side='right'))


########################################

# ATR
md["atr"] = talib.ATR(md["High"], md["Low"], md["Close"], timeperiod=5)

# EMA
md["ema1"] = talib.MA(md["Close"], timeperiod=14, matype=1)
md["ema2"] = talib.MA(md["Close"], timeperiod=14, matype=0)
md["mid"] = (md["ema1"] + md["ema2"]) / 2

md["stdev"] = talib.STDDEV(md["Close"], timeperiod=20, nbdev=1)

price_channel = md[["atr", "stdev"]].max(axis=1)

range_width = 1.1

md["top"] = md["mid"] + range_width * price_channel
md["bottom"] = md["mid"] - range_width * price_channel

# Синяя полоса в середине
# line_m = dict(width=2, color='#59c')
line_m = dict(width=2, color='rgba(0,0,0,0.1)')
fig.add_trace(go.Scatter(x=md.index, y=md["mid"], fill=None, line=line_m))

# print(ind)

# Нижний индикатор
ind = 24 - talib.ADX(md["High"], md["Low"], md["Close"], timeperiod=14)
ind_smooth = talib.MA(ind, timeperiod=2, matype=0)
ind_color = np.where(ind_smooth < 0, 'red', '#0c0')

md["ind_smooth"] = ind_smooth

########################################


current = None
open_trade_price = None
total_profit = 0
total_up = 0
total_down = -0.1
position = 100
stop_cnt = 0
open_sell = 0
open_buy = 0

stop_position = -15

for dt, row in md.copy(deep=True).iterrows():
    md.at[dt, 'trade_buy'] = None
    md.at[dt, 'trade_sell'] = None
    md.at[dt, 'trade_close'] = None
    if current:
        if current == "sell":
            trade_price = min(float(row["Open"]), float(row["top"]))
            profit = (open_trade_price - trade_price) * position
            if profit < stop_position:
                # stop loss
                md.at[dt, 'trade_close'] = trade_price
                total_down += profit
                current = None
                stop_cnt += 1
                # print(f"STOP  {dt:%Y-%m-%d}  {trade_price:6.2f}  {profit:+5.2f}")
                # print()

        if current == "buy":
            trade_price = max(float(row["Open"]), float(row["bottom"]))
            profit = (trade_price - open_trade_price) * position
            if profit < stop_position:
                # stop loss
                md.at[dt, 'trade_close'] = trade_price
                total_down += profit
                current = None
                stop_cnt += 1
                # print(f"STOP  {dt:%Y-%m-%d}  {trade_price:6.2f}  {profit:+5.2f}")
                # print()

        if current == "sell" and row["Low"] < row["mid"]:
            trade_price = min(float(row["Open"]), float(row["mid"]))
            md.at[dt, 'trade_close'] = trade_price
            current = None
            profit = (open_trade_price - trade_price) * position
            total_profit += profit
            if profit > 0:
                total_up += profit
            else:
                total_down += profit
            # print(f"Close {dt:%Y-%m-%d}  {trade_price:6.2f}  {profit:+5.2f}")
            # print()
        if current == "buy" and row["High"] > row["mid"]:
            trade_price = max(float(row["Open"]), float(row["mid"]))
            md.at[dt, 'trade_close'] = trade_price
            current = None
            profit = (trade_price - open_trade_price) * position
            total_profit += profit
            if profit > 0:
                total_up += profit
            else:
                total_down += profit
            # print(f"Close {dt:%Y-%m-%d}  {trade_price:6.2f}  {profit:+5.2f}")
            # print()
    else:
        ind_value = row['ind_smooth']
        if row["High"] > row["top"] and ind_value > 1:
            trade_price = float(row["top"])
            md.at[dt, 'trade_sell'] = trade_price
            current = "sell"
            open_sell += 1
            # print(f"Sell  {dt:%Y-%m-%d}  {trade_price:6.2f}   {ind_value:0.2f}")
            open_trade_price = trade_price
        if row["Low"] < row["bottom"] and ind_value > 1:
            trade_price = float(row["bottom"])
            md.at[dt, 'trade_buy'] = row["bottom"]
            current = "buy"
            open_buy += 1
            # print(f"Buy   {dt:%Y-%m-%d}  {trade_price:6.2f}   {ind_value:0.2f}")
            open_trade_price = trade_price

pf = abs(total_up / total_down)

print(f"Total: {total_profit:+0.2f}")
print(f"Gross Up: {total_up:+0.2f}")
print(f"Gross Down: {total_down:+0.2f}")
print(f"Buy: {open_buy}, sell: {open_sell}, stop: {stop_cnt}")
print(f"Profit Factor: {pf:0.2f}")
print()

fig.add_trace(go.Scatter(
    x=md.index,
    y=md["trade_sell"],
    mode="markers+text",
    # text=md["trade_sell"],
    textposition="top center",
    marker=dict(size=10, color="#f00"),
))
fig.add_trace(go.Scatter(
    x=md.index,
    y=md["trade_buy"],
    mode="markers+text",
    # text=md["trade_buy"],
    textposition="bottom center",
    marker=dict(size=10, color="#0d0"),
))
fig.add_trace(go.Scatter(
    x=md.index,
    y=md["trade_close"],
    mode="markers",
    marker=dict(size=8, color="#000"),
))

# Убрать дыры в оси X
fig.update_xaxes(
    range=[md.index[25], md.index[-1]],
    rangeslider_visible=False,
    rangebreaks=[
        dict(bounds=["sat", "mon"]),  # hide weekends
        # dict(bounds=[16, 9.5], pattern="hour"),  # hide extra hours
        dict(values=[
            "2017-12-25",
            "2018-01-01",
            "2018-01-15",
            "2018-02-19",
            "2018-03-30",
        ])  # hide holidays
    ],
    # tickformat="%Y-%m-%d",
    # ticklabelposition="inside left",
    ticks="outside",
    tickwidth=1,
    tickcolor="#ccc",
)
fig.update_yaxes(
    # fixedrange=True,
    # range=[33, 48],
    row=1,
    ticks="outside",
    tickwidth=1,
    tickcolor="#ccc",
)

fig.update_yaxes(
    fixedrange=True,
    showgrid=True,
    gridcolor="#ddd",
    # side="right",
    row=2,
    ticks="outside",
    tickwidth=1,
    tickcolor="#ccc",
)
fig.update_xaxes(
    # fixedrange=True,
    showgrid=True,
    gridcolor="#ddd",
    row=2,
)


###########################
# Buy Sell Bands

# Top
indicator = go.Scatter(
    x=md.index,
    y=md["top"],
    mode="lines",
    line=dict(color="rgba(0,0,0,0.1)", width=2),
)
fig.add_trace(indicator, 1, 1)

# Bottom
indicator = go.Scatter(
    x=md.index,
    y=md["bottom"],
    mode="lines",
    line=dict(color="rgba(0,0,0,0.1)", width=2),
)
fig.add_trace(indicator, 1, 1)


###########################
# Indicator

# Range Strength
bars = go.Bar(
    x=md.index,
    y=ind_smooth,
    marker_color=ind_color,
    # width=40000000
)
fig.add_trace(bars, row=2, col=1)

# Оформление индикатора
# fig.add_hline(y=0, line=dict(color="#000", width=1), row=2)
# fig.add_hline(y=0.3, line=dict(color="blue", width=1), row=2)
# fig.add_hline(y=-0.3, line=dict(color="red", width=1), row=2)
# fig.add_hrect(y0=0, y1=20, line_width=0, fillcolor="green", opacity=0.1, row=2)
# fig.add_hrect(y0=0, y1=-20, line_width=0, fillcolor="red", opacity=0.1, row=2)

fig.show(config={"displayModeBar": False, "showTips": False})

Total: +49542.17
Gross Up: +49542.17
Gross Down: -15329.26
Buy: 61, sell: 55, stop: 65
Profit Factor: 3.23



In [504]:
# from plotly import offline
# fig.layout.width = 2000
# fig.layout.height = 1000
# offline.plot(fig, filename="filename.html", auto_open=True)